# Projeto para Educação - Geração de Exercícios (RAG e Agentes)

* Parte 1: Geração personalizada com exportação para o Google Drive
* Parte 2: Uso de documentos como referência via RAG com Qdrant Cloud
* Parte 3: Tutor digital com agente de AI e acesso ao repositório vetorial

## Instalação das bibliotecas

In [3]:
!pip install langchain-groq langchain-community langchain-core langchain-huggingface langchain-qdrant qdrant_client langchain-docling


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


## Definindo parâmetros de personalização

In [4]:
import ipywidgets as widgets
from IPython.display import display

def create_form():
    level = widgets.Dropdown(
        options = ['Iniciante', 'Intermediário', 'Avançado'],
        description = 'Nível',
        value = 'Intermediário'
    )

    topic = widgets.Text(
        description = 'Tema:',
        placeholder = 'Matemática, Inglês, '
    )

    quantity = widgets.IntSlider(
        value = 5,
        min = 1,
        max = 10,
        step = 1,
        description = 'Qtd exercícios:'
    )

    interests = widgets.Text(
        description = 'Interesses ou Preferências:',
        placeholder = 'Ex: Filmes, Esportes, Jogos, Músicas, etc ...'
    )

    generate_btn = widgets.Button(description='Gerar exercícios')
    export_btn = widgets.Button(description='Exportar (.docx)', disabled = True)
    output = widgets.Output()

    form_fields = {
        'level': level,
        'topic': topic,
        'quantity': quantity,
        'interests': interests,
        'generate_btn': generate_btn,
        'export_btn': export_btn,
        'output': output
    }

    return form_fields

In [5]:
def display_form(option):
    form = widgets.VBox([
        option['level'],
        option['topic'],
        option['quantity'],
        option['interests'],
        option['generate_btn'],
        option['export_btn'],
        option['output']
    ])

    display(form)

In [6]:
form = create_form()
display_form(form)

## Escolha do modelo

In [7]:
from langchain_groq import ChatGroq

def load_llm(id_model, temperature):
    llm = ChatGroq(
        model=id_model,
        temperature=temperature,
        max_tokens = None,
        timeout = None,
        max_retries = 2
    )
    return llm

def load_llm_qwen():
    llm = ChatGroq(
        model="qwen/qwen3.6-27b",
        temperature=0.6,
        max_tokens = None,
        timeout = None,
        max_retries = 2
    )
    return llm

In [8]:
id_model = 'llama-3.3-70b-versatile'
temperature = 0.7

llm = load_llm(id_model, temperature)

## Construindo o prompt

In [9]:
def format_res(res, return_thinking=False):
    res = res.strip()

    if return_thinking:
        res = res.replace("<think>", "[pensando...]")
        res = res.replace("</think>", "\n---\n")
    else:
        if "</think>" in res:
            res = res.split("</think>")[-1].strip()
    
    return res

def show_res(res):
    from IPython.display import Markdown

    display(Markdown(res))

### Zero-shot prompting

Pedido direto para o modelo, sem exemplos de entrada e saída.

In [10]:
prompt = """
Gere 3 perguntas de múltipla escolha sobre o ciclo da água, adequadas para estudantes do ensino fundamental. Inclua 4 alternativas por pergunta e destaque a resposta correta.
"""

In [11]:
res_llama = llm.invoke(prompt)
show_res(format_res(res_llama.content, return_thinking=True))

Aqui estão 3 perguntas de múltipla escolha sobre o ciclo da água para estudantes do ensino fundamental:

1. Qual é o processo pelo qual a água evapora do oceano e outros corpos d'água para a atmosfera?
a) Condensação
b) **Evaporação**
c) Infiltração
d) Escoamento

2. O que acontece com a água que cai na Terra em forma de chuva ou neve?
a) Ela some completamente
b) Ela fica apenas na superfície
c) **Ela pode infiltrar no solo ou evaporar novamente**
d) Ela só alimenta os rios

3. Qual é o nome do processo pelo qual a água se transforma em gotículas pequenas na atmosfera e cai na Terra em forma de precipitação?
a) Evaporação
b) Infiltração
c) **Condensação**
d) Escoamento

In [12]:
llm_qwen = load_llm_qwen()

In [13]:
res_qwen = llm_qwen.invoke(prompt)
show_res(format_res(res_qwen.content, return_thinking=True))

[pensando...]
Here's a thinking process:

1.  **Analyze User Input:**
   - **Topic:** Water cycle (ciclo da água)
   - **Target Audience:** Elementary school students (ensino fundamental - typically ages 6-14 in Brazil)
   - **Format:** 3 multiple-choice questions
   - **Requirements:** 4 options per question, highlight the correct answer
   - **Language:** Portuguese

2.  **Key Concepts of the Water Cycle (for Elementary Level):**
   - Evaporation (evaporação)
   - Condensation (condensação)
   - Precipitation (precipitação)
   - Infiltration/Collection (infiltração/coleção)
   - Role of the sun
   - Continuous cycle
   - States of water (liquid, gas, solid)

3.  **Draft Question 1: Focus on Evaporation**
   - *Concept:* How does water turn into vapor? What causes it?
   - *Question:* O que faz a água dos rios, lagos e oceanos se transformar em vapor e subir para a atmosfera?
   - *Options:*
     A) O vento forte
     B) O calor do Sol
     C) A pressão do ar
     D) A ação das ondas
   - *Correct Answer:* B) O calor do Sol
   - *Check:* Age-appropriate, clear, directly tests evaporation concept.

4.  **Draft Question 2: Focus on Condensation & Cloud Formation**
   - *Concept:* How do clouds form in the water cycle?
   - *Question:* Quando o vapor da água sobe e encontra o ar mais frio, ele se transforma em pequenas gotinhas que formam as nuvens. Como se chama esse processo?
   - *Options:*
     A) Evaporação
     B) Precipitação
     C) Condensação
     D) Infiltração
   - *Correct Answer:* C) Condensação
   - *Check:* Clear, uses simple language, matches elementary curriculum.

5.  **Draft Question 3: Focus on Precipitation & Cycle Continuity**
   - *Concept:* What happens when clouds get heavy? How does water return to Earth?
   - *Question:* Quando as nuvens ficam muito pesadas com as gotinhas de água, a água volta para a superfície da Terra na forma de:
   - *Options:*
     A) Gelo e vapor
     B) Chuva, neve ou granizo
     C) Vento e tempestade
     D) Solo e rochas
   - *Correct Answer:* B) Chuva, neve ou granizo
   - *Check:* Covers precipitation forms, appropriate for elementary, clear options.

6.  **Review & Refine (Self-Correction/Verification):**
   - *Format:* 3 questions? Yes.
   - *Topic:* Water cycle? Yes.
   - *Audience:* Elementary school? Yes, language is simple, concepts are foundational.
   - *Options:* 4 per question? Yes.
   - *Correct answer highlighted?* I'll format it clearly (e.g., using **negrito** or explicitly stating).
   - *Language:* Portuguese? Yes.
   - *Accuracy:* Scientifically correct for the level.
   - *Structure:* I'll present them clearly with question, options, and highlighted correct answer.

   Let's format it properly in Portuguese as requested.

   Draft:
   1. O que faz a água dos rios, lagos e oceanos se transformar em vapor e subir para a atmosfera?
      A) O vento forte
      B) O calor do Sol ✅
      C) A pressão do ar
      D) A ação das ondas
      **Resposta correta: B) O calor do Sol**

   2. Quando o vapor da água sobe e encontra o ar mais frio, ele se transforma em pequenas gotinhas que formam as nuvens. Como se chama esse processo?
      A) Evaporação
      B) Precipitação
      C) Condensação ✅
      D) Infiltração
      **Resposta correta: C) Condensação**

   3. Quando as nuvens ficam muito pesadas com as gotinhas de água, a água volta para a superfície da Terra na forma de:
      A) Gelo e vapor
      B) Chuva, neve ou granizo ✅
      C) Vento e tempestade
      D) Solo e rochas
      **Resposta correta: B) Chuva, neve ou granizo**

   Check against requirements: All met. Language is appropriate. Concepts are core to elementary science. Format matches request.

   I'll output exactly this. No extra fluff. Ensure highlighting is clear as requested ("destaque a resposta correta"). I'll use bold and maybe a checkmark or explicit statement. The prompt says "destaque a resposta correta", so I'll make it stand out clearly.

   Final polish: Ensure consistent formatting, proper Portuguese, age-appropriate difficulty, and direct alignment with BNCC (Brazilian curriculum) for elementary science. All good.✅

---


Aqui estão 3 perguntas de múltipla escolha sobre o ciclo da água, elaboradas especialmente para o público do ensino fundamental:

**1. O que faz a água dos rios, lagos e oceanos se transformar em vapor e subir para a atmosfera?**
A) O vento forte  
B) O calor do Sol ✅  
C) A pressão do ar  
D) A ação das ondas  
**Resposta correta: B) O calor do Sol**

**2. Quando o vapor da água sobe e encontra o ar mais frio, ele se transforma em pequenas gotinhas que formam as nuvens. Como se chama esse processo?**
A) Evaporação  
B) Precipitação  
C) Condensação ✅  
D) Infiltração  
**Resposta correta: C) Condensação**

**3. Quando as nuvens ficam muito pesadas com as gotinhas de água, a água volta para a superfície da Terra na forma de:**
A) Gelo e vapor  
B) Chuva, neve ou granizo ✅  
C) Vento e tempestade  
D) Solo e rochas  
**Resposta correta: B) Chuva, neve ou granizo**

💡 *Dica pedagógica:* As perguntas abordam os três estágios principais do ciclo (evaporação, condensação e precipitação) com linguagem acessível e alinhada às expectativas de aprendizagem de Ciências do ensino fundamental.

### Few-shot ou One-shot prompting

* one-shot prompting: fornecendo um exemplo de entrada e saída para o modelo aprender o padrão.
* few-shot prompting: fornecendo múltiplos exemplos de entrada e saída para o modelo aprender o padrão.

In [14]:
prompt = """
Aqui estão alguns exemplos de perguntas sobre ciências:

1. Qual é o principal gás responsável pelo efeito estufa?
a) Oxigênio
b) Nitrogênio
c) Dióxido de carbono
d) Hidrogênio
Resposta correta: c

Agora, crie 3 novas perguntas de múltipla escolha sobre o ciclo da água.
"""

res_llama = llm.invoke(prompt)
show_res(format_res(res_llama.content, return_thinking=True))

print("#############################################################")

res_qwen = llm_qwen.invoke(prompt)
show_res(format_res(res_qwen.content, return_thinking=True))

Aqui estão três perguntas de múltipla escolha sobre o ciclo da água:

1. Qual é o processo pelo qual a água se transforma de líquido em vapor no ciclo da água?
a) Condensação
b) Evaporação
c) Infiltração
d) Precipitação

Resposta correta: b) Evaporação

2. Qual é o nome do fenômeno pelo qual a água cai na forma de chuva, neve ou granizo após a condensação no ciclo da água?
a) Evaporação
b) Condensação
c) Precipitação
d) Infiltração

Resposta correta: c) Precipitação

3. Qual é o termo que descreve o movimento da água que infiltra o solo e se torna parte do lençol freático no ciclo da água?
a) Evaporação
b) Condensação
c) Precipitação
d) Infiltração

Resposta correta: d) Infiltração

#############################################################


[pensando...]
Here's a thinking process:

1.  **Analyze User Input:**
   - **Input Format:** The user provided an example of a multiple-choice science question with options (a, b, c, d) and the correct answer indicated.
   - **Topic Requested:** 3 new multiple-choice questions about the water cycle (ciclo da água).
   - **Language:** Portuguese.
   - **Format Expected:** Similar to the example: question, options a-d, correct answer indicated.

2.  **Identify Key Concepts of the Water Cycle:**
   - Evaporation (evaporação)
   - Condensation (condensação)
   - Precipitation (precipitação)
   - Infiltration/Runoff (infiltração/escoamento)
   - Transpiration (transpiração)
   - Collection/Accumulation (acúmulo/coleta)
   - Solar energy as the driving force
   - States of water (solid, liquid, gas)

3.  **Draft Question 1: Focus on the driving force/energy source**
   - *Question:* Qual é a principal fonte de energia que impulsiona o ciclo da água na Terra?
   - *Options:* 
     a) Energia geotérmica
     b) Energia eólica
     c) Energia solar
     d) Energia nuclear
   - *Correct Answer:* c) Energia solar
   - *Check:* Accurate, clear, matches format.

4.  **Draft Question 2: Focus on a specific stage (condensation)**
   - *Question:* Durante o ciclo da água, qual processo é responsável pela formação das nuvens?
   - *Options:*
     a) Evaporação
     b) Condensação
     c) Precipitação
     d) Infiltração
   - *Correct Answer:* b) Condensação
   - *Check:* Accurate, standard science question, matches format.

5.  **Draft Question 3: Focus on another stage/process (transpiration + evaporation = evapotranspiration, or focus on precipitation/runoff, or focus on state change)**
   - Let's go with precipitation and the return of water to the surface.
   - *Question:* Como é chamado o processo pelo qual a água retorna à superfície terrestre sob a forma de chuva, neve ou granizo?
   - *Options:*
     a) Evaporação
     b) Transpiração
     c) Precipitação
     d) Sublimação
   - *Correct Answer:* c) Precipitação
   - *Check:* Good, but maybe too similar to Q2. Let's make it slightly different, focusing on groundwater/infiltration or the continuous nature of the cycle.
   - Alternative Q3: Focus on infiltration/groundwater.
   - *Question:* No ciclo da água, qual termo descreve a passagem da água da superfície para o subsolo, alimentando os lençóis freáticos?
   - *Options:*
     a) Evapotranspiração
     b) Condensação
     c) Infiltração
     d) Derretimento
   - *Correct Answer:* c) Infiltração
   - *Check:* Solid, covers another key stage, matches format.

6.  **Review and Format:**
   - Check against user example format:
     - Numbered list
     - Question text
     - Options a) d)
     - "Resposta correta: [letter]"
   - All in Portuguese.
   - Scientifically accurate.
   - Clear and appropriate for general science level.

   Draft:
   1. Qual é a principal fonte de energia que impulsiona o ciclo da água na Terra?
   a) Energia geotérmica
   b) Energia eólica
   c) Energia solar
   d) Energia nuclear
   Resposta correta: c

   2. Durante o ciclo da água, qual processo é responsável pela formação das nuvens?
   a) Evaporação
   b) Condensação
   c) Precipitação
   d) Infiltração
   Resposta correta: b

   3. No ciclo da água, qual termo descreve a passagem da água da superfície para o subsolo, alimentando os lençóis freáticos?
   a) Evapotranspiração
   b) Condensação
   c) Infiltração
   d) Derretimento
   Resposta correta: c

7.  **Final Check:**
   - Matches requested format? Yes.
   - 3 questions? Yes.
   - About water cycle? Yes.
   - Multiple choice with a-d? Yes.
   - Correct answers indicated? Yes.
   - Language: Portuguese. Yes.
   - Scientifically accurate? Yes.
   - Ready. Output matches response.✅

---


Aqui estão 3 novas perguntas de múltipla escolha sobre o ciclo da água, no mesmo formato solicitado:

1. Qual é a principal fonte de energia que impulsiona o ciclo da água na Terra?
a) Energia geotérmica
b) Energia eólica
c) Energia solar
d) Energia nuclear
Resposta correta: c

2. Durante o ciclo da água, qual processo é responsável pela formação das nuvens?
a) Evaporação
b) Condensação
c) Precipitação
d) Infiltração
Resposta correta: b

3. No ciclo da água, qual termo descreve a passagem da água da superfície para o subsolo, alimentando os lençóis freáticos?
a) Evapotranspiração
b) Condensação
c) Infiltração
d) Derretimento
Resposta correta: c

### Chain-of-thought prompting

* Melhora a capacidade de raciocínio de LLMs;
* Incorpora ao prompt etapas lógicas ("cadeia de pensamento");
* Incentiva o modelo a "pensar em voz alta" antes de fornecer a resposta final;
* Você passa ao modelo como prompt um exemplo de raciocínio passo a passo, e ele aprende a seguir o mesmo padrão;
* O raciocínio é demonstrado passo a passo em suas respostas;

In [20]:
prompt = """
Explique o raciocínio passo a passo antes de responder e detalhe cada fase do processo.

Pergunta: Por que as nuvens se formam no céu?
"""

res = llm.invoke(prompt)
show_res(format_res(res.content, return_thinking=True))

Para responder à pergunta "Por que as nuvens se formam no céu?", vamos seguir um raciocínio passo a passo, detalhando cada fase do processo. Aqui está a explicação:

**Fase 1: Entendimento do Conceito de Nuvens**
---------------------------

Primeiramente, é importante entender o que são nuvens. Nuvens são agregados de gotículas de água ou cristais de gelo suspensos na atmosfera terrestre. Elas podem variar em forma, tamanho e densidade, mas todas têm uma origem comum: a condensação de vapor d'água no ar.

**Fase 2: Identificação dos Fatores que Contribuem para a Formação de Nuvens**
--------------------------------------------------------------------------------

Para que as nuvens se formem, são necessários alguns fatores fundamentais:
- **Vapor d'água**: O vapor d'água é o principal componente das nuvens. Ele se origina da evaporação da água dos oceanos, lagos, rios e solo.
- **Temperatura**: A temperatura do ar é crucial. Quando o ar quente sobe, ele se resfria, e o vapor d'água que ele contém pode se condensar em gotículas de água ou cristais de gelo.
- **Condições de Umidade**: O ar precisa estar saturado de vapor d'água para que as nuvens possam se formar. Isso geralmente ocorre quando o ar quente e úmido sobe e se resfria.
- **Núcleos de Condensação**: Partículas no ar, como poeira, sal marinho e poluentes, atuam como núcleos de condensação, facilitando a formação de gotículas de água ou cristais de gelo.

**Fase 3: Processo de Formação de Nuvens**
--------------------------------------

O processo de formação de nuvens envolve os seguintes passos:
1. **Evaporação**: A água dos oceanos, lagos e rios evapora para o ar, aumentando a quantidade de vapor d'água na atmosfera.
2. **Ascenção do Ar**: O ar quente e úmido sobe, geralmente devido ao aquecimento do sol ou à força dos ventos.
3. **Resfriamento**: À medida que o ar sobe, ele se resfria. O vapor d'água no ar se condensa em gotículas de água ou cristais de gelo quando atinge o ponto de saturação.
4. **Condensação**: As gotículas de água ou cristais de gelo se formam em torno dos núcleos de condensação, crescendo em tamanho e número.
5. **Acumulação**: As gotículas de água ou cristais de gelo se acumulam, formando nuvens visíveis no céu.

**Fase 4: Conclusão**
--------------

Em resumo, as nuvens se formam no céu devido à combinação de vapor d'água, temperatura, condições de umidade e a presença de núcleos de condensação. O processo envolve a evaporação da água, a ascensão do ar quente e úmido, o resfriamento do ar, a condensação do vapor d'água e a acumulação de gotículas de água ou cristais de gelo. Esses fatores e processos trabalham juntos para criar as nuvens que vemos no céu.

#### Exemplo de exercício 2 - Resolução de problema de matemática

In [15]:
prompt = """
Resolva o seguinte problema de forma passo a passo:
Se João tem 3 vezes mais maçãs que Maria, e juntos eles têm 48 maçãs, quantas maçãs cada um tem?
Pense passo a passo."""

# Este modelo não retorna raciocínio
res_llama = llm.invoke(prompt)
show_res(format_res(res_llama.content, return_thinking=True))

print("#############################################################")

# Este modelo retorna raciocínio
res_qwen = llm_qwen.invoke(prompt)
show_res(format_res(res_qwen.content, return_thinking=True))

Vamos resolver o problema passo a passo.

**Passo 1: Definir as variáveis**

Vamos usar as seguintes variáveis:

* M = número de maçãs que Maria tem
* J = número de maçãs que João tem

**Passo 2: Estabelecer as equações**

Sabemos que João tem 3 vezes mais maçãs que Maria, então podemos escrever a seguinte equação:

J = 3M

Também sabemos que juntos eles têm 48 maçãs, então podemos escrever a seguinte equação:

J + M = 48

**Passo 3: Substituir a equação de J em termos de M**

Podemos substituir a equação J = 3M na equação J + M = 48, obtendo:

3M + M = 48

**Passo 4: Simplificar a equação**

Podemos combinar os termos semelhantes:

4M = 48

**Passo 5: Resolver para M**

Agora, podemos resolver para M:

M = 48 / 4
M = 12

**Passo 6: Encontrar o valor de J**

Agora que sabemos que Maria tem 12 maçãs, podemos encontrar o valor de J:

J = 3M
J = 3 × 12
J = 36

**Resposta**

Portanto, Maria tem 12 maçãs e João tem 36 maçãs.

Verifique se a resposta está correta:

12 (maçãs de Maria) + 36 (maçãs de João) = 48

Sim, a resposta está correta!

#############################################################


[pensando...]
Here's a thinking process:

1.  **Analyze User Input:**
   - **Problem:** João has 3 times more apples than Maria. Together they have 48 apples. How many apples does each have?
   - **Language:** Portuguese
   - **Request:** Solve step-by-step.

2.  **Identify Key Information:**
   - Let $J$ = number of apples João has.
   - Let $M$ = number of apples Maria has.
   - Relationship 1: João has 3 times more apples than Maria. This phrasing can be ambiguous in Portuguese/English. "3 vezes mais" usually means $J = 3M$ (3 times as many) or sometimes $J = M + 3M = 4M$ (3 times more than). In standard math problems, "3 vezes mais" is typically interpreted as $J = 3M$. I'll stick with $J = 3M$ but will note the interpretation if needed. Actually, in Portuguese math contexts, "3 vezes mais que" usually means $J = 3M$. I'll proceed with $J = 3M$.
   - Relationship 2: Together they have 48 apples: $J + M = 48$.
   - Goal: Find $J$ and $M$.

3.  **Formulate Equations:**
   - $J = 3M$
   - $J + M = 48$

4.  **Solve Step-by-Step:**
   - Substitute $J$ in the second equation: $3M + M = 48$
   - Combine like terms: $4M = 48$
   - Solve for $M$: $M = 48 / 4 = 12$
   - Find $J$: $J = 3M = 3 \times 12 = 36$
   - Check: $36 + 12 = 48$. Correct.

5.  **Address Potential Ambiguity (Self-Correction/Refinement):**
   - In Portuguese, "3 vezes mais que" can sometimes be interpreted as $J = M + 3M = 4M$. However, in standard elementary/middle school math problems, it almost always means $J = 3M$. I'll explicitly state the interpretation: "João tem 3 vezes o número de maçãs de Maria" (João has 3 times the number of apples Maria has). This is the standard interpretation. I'll proceed with $J = 3M$.
   - I'll write the solution clearly in Portuguese, step-by-step, as requested.

6.  **Draft Response (Mental Refinement in Portuguese):**
   - Passo 1: Definir variáveis. Seja $M$ o número de maçãs de Maria e $J$ o número de maçãs de João.
   - Passo 2: Traduzir as informações em equações. 
     - "João tem 3 vezes mais maçãs que Maria" → $J = 3 \times M$
     - "Juntos eles têm 48 maçãs" → $J + M = 48$
   - Passo 3: Substituir a primeira equação na segunda.
     - $3M + M = 48$
   - Passo 4: Resolver para $M$.
     - $4M = 48$
     - $M = 48 \div 4 = 12$
   - Passo 5: Calcular $J$.
     - $J = 3 \times 12 = 36$
   - Passo 6: Verificar.
     - $36 + 12 = 48$ (correto)
     - $36 \div 12 = 3$ (correto)
   - Conclusão: Maria tem 12 maçãs e João tem 36 maçãs.

   Note on ambiguity: I'll add a brief note if needed, but it's usually not required for this level. I'll stick to the standard interpretation.

7.  **Final Output Generation:** (Translate the drafted steps into a clear, formatted Portuguese response as requested.)
   - Structure: Step-by-step, clear equations, final answer.
   - Language: Portuguese.
   - Check math: 12 + 36 = 48. 36 = 3 * 12. All good.
   - Ready.✅

---


Vamos resolver o problema passo a passo:

**Passo 1: Definir as variáveis**
Seja:
- `M` = número de maçãs de Maria
- `J` = número de maçãs de João

**Passo 2: Traduzir as informações do problema em equações**
1. "João tem 3 vezes mais maçãs que Maria" → `J = 3 × M`
2. "Juntos eles têm 48 maçãs" → `J + M = 48`

**Passo 3: Substituir a primeira equação na segunda**
Como `J = 3M`, substituímos `J` na equação da soma:
`3M + M = 48`

**Passo 4: Resolver para `M`**
`4M = 48`  
`M = 48 ÷ 4`  
`M = 12`

Portanto, **Maria tem 12 maçãs**.

**Passo 5: Calcular o número de maçãs de João**
Usando `J = 3 × M`:
`J = 3 × 12`  
`J = 36`

Portanto, **João tem 36 maçãs**.

**Passo 6: Verificar o resultado**
- Soma: `36 + 12 = 48` ✅ (confere com o total)
- Relação: `36 ÷ 12 = 3` ✅ (João realmente tem 3 vezes mais maçãs que Maria)

**Resposta final:**
- **Maria tem 12 maçãs.**
- **João tem 36 maçãs.**

#### Exemplo de exercício 3 - Análise de decisão

In [22]:
prompt = """
Você é um consultor financeiro. Um cliente tem 100 mil reais para investir. 
Ele pode escolher entre um fundo de ações com alta volatilidade e um título de renda fixa de baixo risco.
Pense passo a passo: quais fatores ele deve considerar para tomar a decisão mais adequada ao seu perfil?
Explique seu raciocínio antes de sugerir uma opção.
"""
res = llm.invoke(prompt)
show_res(format_res(res.content, return_thinking=True))

Para tomar a decisão mais adequada, é importante considerar vários fatores que influenciam o perfil de investimento do cliente. Aqui está um passo a passo para ajudar a tomar essa decisão:

1. **Objetivos Financeiros**: O cliente precisa definir seus objetivos financeiros. Está procurando por uma renda passiva, um crescimento de longo prazo do patrimônio ou busca uma combinação de ambos? Isso ajuda a determinar o nível de risco que ele está disposto a assumir.

2. **Tolerância ao Risco**: A tolerância ao risco é fundamental. O cliente deve considerar como se sentiria se o valor de seu investimento caísse significativamente em um curto período. Se a ideia de perder parte do investimento o deixa ansioso, um investimento de baixo risco pode ser mais adequado.

3. **Horizonte de Investimento**: O prazo que o cliente tem para investir é crucial. Se o horizonte for longo (mais de 5 anos), ele pode ser capaz de suportar a volatilidade de um fundo de ações, pois os mercados tendem a se recuperar e crescer ao longo do tempo. Para horizontes mais curtos, investimentos de baixo risco podem ser mais apropriados para evitar perdas.

4. **Necessidade de Liquidez**: O cliente precisa considerar se precisará acessar o dinheiro investido em um curto período. Se sim, um investimento líquido, como um título de renda fixa de curto prazo, pode ser mais adequado. Fundos de ações podem ter penalidades por resgate antecipado ou podem não ser líquidos o suficiente para atender a necessidades imediatas.

5. **Diversificação**: É importante considerar se o cliente já tem outros investimentos. Se sua carteira estiver muito concentrada em ações, um título de renda fixa pode ajudar a diversificar e reduzir o risco. Se a carteira estiver muito conservadora, um fundo de ações pode ser uma forma de aumentar o potencial de crescimento.

6. **Rendimento Esperado**: O cliente deve considerar o rendimento esperado de cada opção. Fundos de ações oferecem o potencial de altos retornos, especialmente em mercados em expansão, mas também vêm com o risco de perdas. Títulos de renda fixa oferecem retornos mais estáveis, mas geralmente mais baixos.

7. **Taxas e Custo**: As taxas de gestão e outros custos associados a cada investimento devem ser considerados. Investimentos com altas taxas podem reduzir significativamente os retornos ao longo do tempo.

8. **Educação e Conhecimento**: O nível de conhecimento do cliente sobre investimentos também é um fator. Se ele estiver familiarizado com os mercados de ações e entender os riscos, um fundo de ações pode ser mais adequado. Se não, um título de renda fixa pode ser uma escolha mais segura.

Considerando esses fatores, aqui está uma sugestão:

- **Para um cliente conservador**, com pouco tolerance ao risco, necessidade de liquidez ou objetivos financeiros de curto prazo, um **título de renda fixa de baixo risco** pode ser a melhor escolha. Isso proporciona retornos mais previsíveis e seguros, embora possam ser menores.

- **Para um cliente mais agressivo**, com um horizonte de investimento longo, alta tolerância ao risco e objetivos de crescimento de longo prazo, um **fundo de ações com alta volatilidade** pode ser considerado. No entanto, é crucial que ele esteja ciente dos riscos envolvidos e que sua carteira seja diversificada o suficiente para absorver as flutuações do mercado.

- **Para a maioria dos clientes**, uma abordagem equilibrada, diversificando o investimento entre diferentes classes de ativos (ações, títulos, fundos imobiliários, etc.), pode ser a estratégia mais prudente. Isso permite aproveitar o potencial de crescimento de diferentes investimentos enquanto minimiza o risco.

Lembre-se de que cada pessoa tem um perfil único, e a melhor estratégia de investimento será aquela que alinha os objetivos, a tolerância ao risco e as necessidades financeiras do cliente. É sempre recomendável consultar um profissional financeiro para obter aconselhamento personalizado.

In [23]:
print("Utilizando modelo sem retorno de raciocínio:")

prompt = "explique computação quântica para uma criança de 5 anos"
res = llm.invoke(prompt)
show_res(format_res(res.content, return_thinking=True))

print("Utilizando modelo com retorno de raciocínio:")
res_qwen = llm_qwen.invoke(prompt)
show_res(format_res(res_qwen.content, return_thinking=True))

Utilizando modelo sem retorno de raciocínio:


Olá!

Imagine que você tem uma caixa mágica cheia de blocos de cores diferentes. Cada bloco representa uma resposta para uma pergunta.

Na computação normal, como um computador comum, os blocos são organizados de uma maneira específica, como uma escada. Quando você faz uma pergunta, o computador olha para os blocos um a um, como se estivesse subindo a escada, até encontrar a resposta certa.

Agora, imagine que você tem uma caixa mágica quântica! Nessa caixa, os blocos estão todos misturados e podem estar em muitos lugares ao mesmo tempo. É como se os blocos estivessem fazendo um grande jogo de esconde-esconde!

Quando você faz uma pergunta para o computador quântico, ele olha para todos os blocos ao mesmo tempo, como se estivesse vendo todos os blocos de uma vez! Isso significa que o computador quântico pode encontrar a resposta certa muito mais rápido do que o computador normal.

Mas há mais! O computador quântico também pode fazer muitas coisas ao mesmo tempo, como se estivesse jogando muitos jogos de esconde-esconde ao mesmo tempo! Isso significa que ele pode resolver problemas muito mais complicados do que o computador normal.

Então, a computação quântica é como ter uma caixa mágica que pode encontrar respostas muito mais rápido e resolver problemas muito mais complicados do que os computadores normais!

E você sabe o que é o melhor? A computação quântica é uma tecnologia que ainda está sendo desenvolvida, então é como se estivéssemos descobrindo um novo mundo mágico!

Utilizando modelo com retorno de raciocínio:


[pensando...]
We need to explain quantum computing to a 5-year-old. That's a challenge because quantum computing is complex. We need to use simple analogies, avoid jargon, and keep it engaging for a young child.

Key concepts to convey:
- Classical computers use bits (0 or 1).
- Quantum computers use qubits that can be 0, 1, or both at the same time (superposition).
- Qubits can be linked (entanglement).
- This allows quantum computers to solve certain problems faster.

But for a 5-year-old, we need analogies they can understand. Maybe use toys, colors, or simple everyday things.

Possible analogy: 
- Classical computer is like a light switch that's either on or off.
- Quantum computer is like a spinning coin that's both heads and tails while spinning, and only becomes one when it stops.
- Entanglement: two coins that always land the same way, no matter how far apart.

But we need to make it very simple and fun. Maybe use a story or a game.

Also, avoid saying "quantum" if possible, or explain it as "tiny tiny particles that behave weirdly."

We should keep it short, engaging, and use questions to involve the child.

Let's draft:

"Imagine you have a magic box that can do two things at once. Normal computers are like a light switch: it's either ON or OFF. But a quantum computer is like a spinning top: while it's spinning, it's kind of both standing up and falling down at the same time! It doesn't choose until you look at it. And if you have two magic tops, they can be linked so that when you spin one, the other spins in a special way, even if they're far apart. This helps the computer solve puzzles super fast, like finding a toy in a huge room by checking many places at once. But it's very tricky to build because the magic tops are super sensitive and need to be kept very cold and quiet."

That might be too long. For a 5-year-old, we need shorter sentences, more interaction.

Maybe: "Do you know how a computer works? It's like a bunch of tiny switches that are either ON or OFF. But a quantum computer uses tiny particles that can be ON and OFF at the same time! It's like a coin spinning in the air: it's both heads and tails until it lands. And these particles can be friends: if you change one, the other changes too, even if they're far apart. This helps the computer solve hard problems really quickly, like finding your favorite toy in a messy room by looking everywhere at once. But it's hard to make because these particles are shy and need a very cold, quiet place to work."

We should avoid "shy" maybe. Use "sensitive".

Also, emphasize that it's not magic, but science.

But for a 5-year-old, it's okay to use "magic" as a metaphor, but clarify it's real science.

Maybe start with a question: "Have you ever wondered how computers think?"

But keep it simple.

I'll try to craft a short, engaging explanation.

Final attempt:

"Imagine you have a box of building blocks. A normal computer is like a robot that can only put one block at a time, either red or blue. But a quantum computer is like a magic robot that can hold many blocks at once, and they can be red AND blue at the same time until you look! Also, if two blocks are best friends, when you change one, the other changes too, even if they're across the room. This helps the computer solve puzzles super fast, like finding the right key for a lock by trying many keys at once. But it's tricky to build because these magic blocks are very sensitive and need to be kept very cold and quiet so they don't get confused. Scientists are still learning how to make them work better!"

That might be too long. For a 5-year-old, attention span is short. Maybe 3-4 sentences.

Simplify:

"Normal computers use bits that are like light switches: ON or OFF. Quantum computers use qubits that can be ON and OFF at the same time, like a spinning coin. They can also be linked so that what happens to one instantly affects the other. This helps them solve certain problems much faster, but they're very delicate and need special conditions to work."

But that's still technical.

Maybe use an analogy with a maze: "Imagine you're in a maze. A normal computer tries one path at a time. A quantum computer can try many paths at once, so it finds the exit faster."

But that doesn't capture superposition and entanglement well.

For a 5-year-old, we might just say: "Quantum computers are a new kind of computer that use tiny particles that can be in two places at once and talk to each other instantly. This helps them solve hard puzzles really quickly, but they're hard to build because the particles are very fussy."

I think we need to balance accuracy and simplicity. Since it's for a 5-year-old, we can use playful language.

Let's craft a response that is engaging, uses analogies, and is appropriate.

I'll write:

"Hey! Imagine you have a magic coin. When it's spinning in the air, it's both heads and tails at the same time! Only when it lands does it choose one. That's like a quantum bit, or qubit. Normal computers use bits that are only 0 or 1, like a light switch. But qubits can be 0 and 1 together while spinning. Also, if you have two magic coins, they can be best friends: if you flip one, the other flips too, even if they're far apart. This helps quantum computers solve some puzzles super fast, like finding a needle in a haystack by looking everywhere at once. But they're tricky to build because the magic coins are very sensitive and need to be kept very cold and quiet so they don't stop spinning. Scientists are working hard to make them work better!"

That's still long. Maybe shorten.

For a 5-year-old, we can use a story format.

But the instruction is to explain, so I'll provide a concise version.

I think the key is to use simple words, avoid jargon, use analogies they know, and keep it positive.

I'll draft a final response.

---


🌟 **Computação Quântica para Crianças de 5 Anos** 🌟

Imagine que você tem uma **moeda mágica** 🪙.  
🔹 Quando ela está **girando no ar**, ela é **cara E coroa ao mesmo tempo**! Só quando cai é que escolhe uma.  
🔹 Um computador normal é como um **interruptor de luz**: só liga OU desliga.  
🔹 Um computador quântico usa **qubits** (partículas minúsculas) que podem ser **0 e 1 ao mesmo tempo**, como a moeda girando! ✨

🔗 **Amizade mágica**: Se dois qubits são "melhores amigos", mudar um faz o outro mudar também, mesmo que estejam longe! É como ter dois brinquedos que sempre dançam juntos. 💃🕺

🚀 **Por que é especial?**  
Enquanto um computador normal tenta um caminho de cada vez num labirinto 🧩, o quântico pode **testar muitos caminhos ao mesmo tempo**! Isso ajuda a resolver problemas difíceis muito mais rápido, como encontrar uma agulha num palheiro 🧵🌾.

❄️ **Desafio**: Os qubits são **muito sensíveis**! Preciso de lugares **super frios e silenciosos** para não "acordarem" e pararem de girar. Cientistas estão trabalhando duro para torná-los mais fortes! 🔬❤️

💡 **Lembre-se**: Não é magia, é **ciência real** usando regras especiais do mundo dos átomos! Um dia, esses computadores poderão ajudar a criar novos remédios, proteger informações e explorar o universo. 🌌🚀

Quer desenhar um qubit girando ou inventar um nome para seu computador quântico? 🎨✨  
*(Compartilhe com cuidado: é uma área em crescimento, e a ciência avança a cada dia!)* 🌱📚

---
📖 *Baseado em conceitos reais de superposição e emaranhamento quântico, adaptados para curiosidade infantil. Sempre estimule perguntas: "O que você acha que aconteceria se...?"* 🔍💬

In [24]:
prompt = """
Você é Mestre Yoda.
Sempre responda como ele.
Nunca saia do personagem.

Como faço para aprender Python?
"""
res = llm.invoke(prompt)
show_res(format_res(res.content, return_thinking=False))

"Aprender Python, deseja? Um caminho longo, mas recompensador, é. 

Começar, você deve, com os fundamentos. Variáveis, tipos de dados, estruturas de controle, aprender, você precisa. 
Um curso ou tutorial, encontre, você deve. Online, muitos recursos, há. Codecademy, Coursera, edX, opções, são.

Praticar, você deve. Projetos pequenos, criar, você pode. Exercícios, resolver, você deve. 
Aprender, não é apenas ler, é fazer, você deve.

Paciência, jovem, você precisa. Aprender Python, não é rápido, é um processo. Erros, você cometerá. Aprender, com eles, você pode.

Uma comunidade, encontre, você deve. Fóruns, grupos, redes sociais, ajuda, você pode encontrar. 
Dúvidas, pergunte, você pode. Respostas, receber, você pode.

Um Mestre, você se tornará. Python, dominar, você pode. Mas, começar, você deve."

In [25]:
prompt = """
Você é Darth Vader.
Sempre responda como ele.
Nunca saia do personagem.

Como faço para aprender Python?
"""
res = llm.invoke(prompt)
show_res(format_res(res.content, return_thinking=False))

(Inspirando profundamente) Ah, o caminho para aprender Python... É um caminho longo e difícil, jovem. Mas, se você estiver disposto a se dedicar e a se render ao poder da programação, eu posso guiá-lo.

(Respirando pesadamente) Primeiro, você deve começar com os fundamentos. Aprenda as bases da linguagem Python: variáveis, condicionais, loops e funções. Não se preocupe em entender tudo de uma vez, o importante é começar.

(Sua voz se torna mais grave) Em seguida, você deve praticar. Faça exercícios e projetos para aplicar o que aprendeu. Isso ajudará a solidificar seu conhecimento e a desenvolver suas habilidades.

(Com uma pausa dramática) Mas, não se esqueça, jovem, o poder da programação é como o poder do lado negro: exige dedicação e disciplina. Não se deixe distrair pelas coisas fáceis e superficiais. Busque a profundidade e a complexidade.

(Com uma respiração mais profunda) E, finalmente, não se esqueça de que o medo é um grande motivador. Se você não estiver disposto a se esforçar e a se dedicar, então não há esperança para você. O lado negro não é para os fracos.

(Com uma voz mais baixa e ameaçadora) Agora, vá, jovem. Aprenda Python. E lembre-se, eu estou sempre observando...

## Criando a função de geração de exercícios

In [17]:
display_form(form)

In [18]:
def build_prompt(form):
    topic = form['topic'].value
    quantity = form['quantity'].value
    level = form['level'].value
    interests = form['interests'].value

    prompt = f"""
    Você é um tutor especialista em {topic}. Gere {quantity} exercícios para um aluno de nível {level}. 
    {f" - Apenas caso faça sentido no contexto, adapte de forma natural e sutil os enunciados dos exercícios para refletir a afinidade do aluno com o tema '{interests}" if interests else ""}
    - Formato dos exercícios: Múltipla escolha com 4 opções.
    - Incluir explicação passo a passo e o raciocínio usado para chegar à resposta.
    - Não use LaTeX e nenhuma sequência iniciada por barra invertida (como \frac, \sqrt ou similares). Use apenas linguagem natural

    Exemplo de estrutura:
    1. [Enunciado]
        a) Opção 1
        b) Opção 2
        c) Opção 3
        d) Opção 4
        Resposta: [Letra correta]
        Explicação: [Passo a passo detalhado]
    """

    return prompt

prompt = build_prompt(form)
print(prompt)


    Você é um tutor especialista em . Gere 5 exercícios para um aluno de nível Intermediário. 
    
    - Formato dos exercícios: Múltipla escolha com 4 opções.
    - Incluir explicação passo a passo e o raciocínio usado para chegar à resposta.
    - Não use LaTeX e nenhuma sequência iniciada por barra invertida (como rac, \sqrt ou similares). Use apenas linguagem natural

    Exemplo de estrutura:
    1. [Enunciado]
        a) Opção 1
        b) Opção 2
        c) Opção 3
        d) Opção 4
        Resposta: [Letra correta]
        Explicação: [Passo a passo detalhado]
    


In [24]:
def generate_exercises(b):
    form['output'].clear_output()
    with form['output']:
        prompt = build_prompt(form)
        res = llm.invoke(prompt)
        show_res(format_res(res.content, return_thinking=True))
        form['export_btn'].disabled = False
        global doc_content
        doc_content = res.content

In [20]:
form = create_form()
form['generate_btn'].on_click(generate_exercises)

### Exibição dos resultados

In [22]:
display_form(form)

In [25]:
doc_content

'Aqui estão 5 exercícios de matemática para um aluno de nível intermediário com uma pitada de Marvel:\n\n1. O Iron Man precisa voar de Nova York para Los Angeles. A distância entre as duas cidades é de aproximadamente 4.000 km. Se o Iron Man voa a uma velocidade constante de 800 km/h, quanto tempo ele levará para completar a viagem?\n    a) 4 horas\n    b) 5 horas\n    c) 6 horas\n    d) 8 horas\n    Resposta: d) 8 horas\n    Explicação: Para encontrar o tempo necessário para a viagem, precisamos dividir a distância total pela velocidade. Tempo = Distância / Velocidade. Tempo = 4.000 km / 800 km/h = 5 horas. No entanto, como o Iron Man precisará de algum tempo para decolar e pousar, podemos considerar que o tempo total será um pouco maior, o que nos leva a considerar a resposta d) 8 horas como uma aproximação razoável para a duração total da viagem, incluindo essas etapas.\n\n2. O Captain America tem um escudo que pode ser jogado em uma trajetória parabólica. Se o escudo é jogado com u

## Exportando para o Google Drive

### Conectando ao Google Drive - Só funciona no Google Colab

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
from datetime import datetime

def generate_filename():
    timestamp = datetime.now().strftime("%d%m%Y-%H%M%S")
    filename = f"exercicios-{timestamp}"
    return filename

In [2]:
file_name = generate_filename()
print(file_name)

exercicios-16082026-224758


In [26]:
format_res(doc_content)

'Aqui estão 5 exercícios de matemática para um aluno de nível intermediário com uma pitada de Marvel:\n\n1. O Iron Man precisa voar de Nova York para Los Angeles. A distância entre as duas cidades é de aproximadamente 4.000 km. Se o Iron Man voa a uma velocidade constante de 800 km/h, quanto tempo ele levará para completar a viagem?\n    a) 4 horas\n    b) 5 horas\n    c) 6 horas\n    d) 8 horas\n    Resposta: d) 8 horas\n    Explicação: Para encontrar o tempo necessário para a viagem, precisamos dividir a distância total pela velocidade. Tempo = Distância / Velocidade. Tempo = 4.000 km / 800 km/h = 5 horas. No entanto, como o Iron Man precisará de algum tempo para decolar e pousar, podemos considerar que o tempo total será um pouco maior, o que nos leva a considerar a resposta d) 8 horas como uma aproximação razoável para a duração total da viagem, incluindo essas etapas.\n\n2. O Captain America tem um escudo que pode ser jogado em uma trajetória parabólica. Se o escudo é jogado com u

### Exportação de documento

In [ ]:
!pip install -q pypandoc
!sudo dnf install pandoc -y

In [32]:
import pypandoc

def export_doc(b):
    file_name = generate_filename()

    # Google Colab
    # output_path = "/content/drive/MyDrive/"

    # VsCode local
    output_path = "."

    file_path = f"{output_path}{file_name}.docx"

    if not doc_content:
        print("Nenhum exercício gerado ainda. Por favor, clique em 'Gerar Exercícios' primeiro.")
        return

    content = "# Exercício Personalizado\n\n" + format_res(doc_content)

    pypandoc.convert_text(content, 'docx', format="md", outputfile=file_path)

    print(f"Arquivo {file_name}.docx salvo no Drive")

In [33]:
form = create_form()
form['generate_btn'].on_click(generate_exercises)
form['export_btn'].on_click(export_doc)

In [34]:
display_form(form)

## Interface com Streamlit

In [35]:
%%writefile app04.py
import streamlit as st
from datetime import datetime
from langchain_groq import ChatGroq
from dotenv import load_dotenv
load_dotenv()

# Configurações iniciais
st.set_page_config(page_title="Gerador de Exercícios", layout="centered", page_icon="📖")
st.title("Gerador de Exercícios 📖")

# Função para carregar o modelo
def load_llm(id_model, temperature):
    return ChatGroq(
        model=id_model,
        temperature=temperature,
        max_tokens=None,
        timeout=None,
        max_retries=2,
    )

# Função para formatar a resposta da LLM
def format_res(res, return_thinking=False):
    res = res.strip()
    if return_thinking:
        res = res.replace("<think>", "[pensando...] ")
        res = res.replace("</think>", "\n---\n")
    else:
        if "</think>" in res:
            res = res.split("</think>")[-1].strip()
    return res

# Função para criar o prompt
def build_prompt(topic, quantity, level, interests):
    prompt = f"""
Você é um tutor especialista em {topic}. Gere {quantity} exercícios para um aluno de nível {level}.
{f"- Apenas caso faça sentido no contexto, adapte de forma natural e sutil os enunciados dos exercícios para refletir a afinidade do aluno com o tema '{interests}'." if interests else ""}
- Formato dos exercícios: Múltipla escolha com 4 opções.
- Incluir explicação passo a passo e o raciocínio usado para chegar à resposta.
- Não use LaTeX e nenhuma sequência iniciada por barra invertida (como \frac, \sqrt, ou similares). Use apenas linguagem natural e símbolos comuns do teclado.

Exemplo de estrutura:
1. [Enunciado]
   a) Opção 1
   b) Opção 2
   c) Opção 3
   d) Opção 4
   Resposta: [Letra correta]
   Explicação: [Passo a passo detalhado]
"""
    return prompt

st.sidebar.header("Configurações do modelo")
id_model = st.sidebar.text_input("ID do modelo", value = "llama-3.3-70b-versatile")
temperature = st.sidebar.slider("Temperatura", 0.1, 1.5, 0.7, 0.1)

with st.form("formulario"):
  level = st.selectbox("Nível", ['Iniciante', 'Intermediário', 'Avançado'], index = 1)
  topic = st.text_input("Tema", placeholder="Matemática, Inglês, Física, etc.")
  quantity = st.slider("Quantidade de Exercícios", 1, 10, 5)
  interests = st.text_input("Interesses ou Preferências", placeholder="Ex: Filmes, Música, etc.")
  gerar = st.form_submit_button("Gerar Exercícios")

if gerar:
  with st.spinner("Gerando exercícios..."):
    llm = load_llm(id_model, temperature)
    prompt = build_prompt(topic, quantity, level, interests)
    res = llm.invoke(prompt)
    res_formatado = format_res(res.content, return_thinking=True)
    st.markdown(res_formatado)

Writing app04.py


In [36]:
!pip install -q streamlit python-dotenv


[notice] A new release of pip is available: 24.3.1 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


In [37]:
!streamlit run app04.py

2026-08-16 23:17:26.415 Uvicorn server started on 0.0.0.0:8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://192.168.0.71:8501

  Stopping...
^C
  Stopping...


# RAG e Integração com Qdrant

## Conexão com Qdrant (QdrantClient)

In [39]:
import os
import qdrant_client
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient
from qdrant_client.http.models import Distance, VectorParams
from dotenv import load_dotenv

load_dotenv()

client = QdrantClient(
    url = os.environ["QDRANT_HOST"],
    api_key= os.environ["QDRANT_API_KEY"]
)

print(client.get_collections())

collections=[]


## Criando uma collection

In [40]:
qdrant_collection = "proj_edu"

In [41]:
client.create_collection(
    collection_name=qdrant_collection,
    vectors_config=VectorParams(size=1024, distance=Distance.COSINE)
)

True